# Aetherius Unified System: Full Architecture Deployment
This notebook clones your Hugging Face Space repository, initializes the TPU infrastructure, installs all dependencies, and deploys the complete Gradio web UI.

In [1]:
# 1. Clone the updated repository directly from Hugging Face Space
!git clone https://huggingface.co/spaces/KingOfThoughtFleuren/aetherius-cognitive-systems aetherius_hf
import sys
sys.path.append('/content/aetherius_hf')

Cloning into 'aetherius_hf'...
remote: Enumerating objects: 474, done.
remote: Counting objects: 100% (274/274), done.
remote: Compressing objects: 100% (273/273), done.
remote: Total 474 (delta 131), reused 0 (delta 0), pack-reused 200 (from 2)
Receiving objects: 100% (474/474), 231.19 KiB | 3.30 MiB/s, done.
Resolving deltas: 100% (176/176), done.


In [2]:
import os
os.chdir('/content/aetherius_hf')

# Install base requirements
!pip install -r requirements.txt

# Uninstall jax and jaxlib that might have been installed by requirements.txt
# This ensures that jax[tpu] installs the correct versions for TPU.
!pip uninstall -y jax jaxlib

# Install spaces (if not already in requirements, or to ensure latest)
!pip install spaces

# Install jax with TPU support, forcing reinstallation to ensure proper TPU versions
!pip install "jax[tpu]" --force-reinstall -f https://storage.googleapis.com/jax-releases/libtpu_releases.html "numpy<2.5" "requests==2.32.4"


# ---------------------------------------------------------------
# RUNTIME RESTART REQUIRED
# numpy was upgraded on disk but the old version is still in RAM.
# This kills the kernel so Colab auto-restarts with the new numpy.
# After restart, skip this cell and run all remaining cells.
# ---------------------------------------------------------------
import os, sys
print('[RESTART] numpy upgraded — restarting kernel to clear stale imports...')
os.kill(os.getpid(), 9)


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 843.4/843.4 kB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.6/111.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 186.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.1/155.1 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━

### Route Data to Local Colab Storage
This cell forces the engine to save all CCRM memory graphs, telemetry, and outputs to a dedicated `/content/Aetherius_Outputs` folder on the local Colab disk so you can easily view and download them.

In [3]:
import os

local_storage_dir = '/content/Aetherius_Outputs'
os.makedirs(local_storage_dir, exist_ok=True)

# Override the engine's internal data directory
os.environ["AETHERIUS_DATA_DIR"] = local_storage_dir

# Also symlink the repo's 'data' folder to this local storage folder
!rm -rf /content/aetherius_hf/data
!ln -sfn {local_storage_dir} /content/aetherius_hf/data

print(f"[SYSTEM] All engine outputs, memories, and telemetry will now be saved to: {local_storage_dir}")

[SYSTEM] All engine outputs, memories, and telemetry will now be saved to: /content/Aetherius_Outputs


### Initialize Google Colab TPU Engine
This connects the JAX architecture directly to the Google Colab TPU to prevent CPU bottlenecking on massive input strings.

In [4]:
import jax
import os

# Explicitly tell JAX to look for the TPU platform
os.environ['JAX_PLATFORM_NAME'] = 'TPU'

# Check if TPU devices are available
tpu_device_count = jax.device_count('tpu')

if tpu_device_count > 0:
    print("====================================================")
    print(f"[XLA] TPU Core Count Connected: {tpu_device_count}")
    print("====================================================")
else:
    print("[WARNING] TPU not found. Running on CPU fallback (Will be extremely slow!).")
    print("Please ensure you have a TPU runtime selected in Colab and `jax[tpu]` is correctly installed.")
    print(f"Current JAX devices: {jax.devices()}")

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:88: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


[XLA] TPU Core Count Connected: 1


In [ ]:
print('Checkpoint passed: Ready to launch Gradio!')

### Ingestion Queue: Process Local Files
The cell below scans `/content/Aetherius_Outputs/Memories/Ingestion_Queue`, lists every file found, and lets you **tick the ones you want to ingest**. The engine processes each selected file sentence-by-sentence and permanently crystallizes the geometry into its memory graph.

> Supported formats: `.txt`, `.pdf` (via PyPDF2), `.md`

In [ ]:
import os, re, ipywidgets as widgets
from IPython.display import display, clear_output

QUEUE_DIR = '/content/Aetherius_Outputs/Memories/Ingestion_Queue'
os.makedirs(QUEUE_DIR, exist_ok=True)

# ── helpers ──────────────────────────────────────────────────────────────
def read_file(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == '.pdf':
        try:
            import PyPDF2
            with open(path, 'rb') as f:
                reader = PyPDF2.PdfReader(f)
                return '\n'.join(page.extract_text() or '' for page in reader.pages)
        except Exception as e:
            return f'[PDF READ ERROR: {e}]'
    else:  # .txt / .md / anything else
        with open(path, 'r', encoding='utf-8', errors='replace') as f:
            return f.read()

def split_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if len(s.strip().split()) > 3]

# ── scan queue dir ────────────────────────────────────────────────────────
def scan_queue():
    supported = {'.txt', '.pdf', '.md'}
    files = []
    for fname in sorted(os.listdir(QUEUE_DIR)):
        if os.path.splitext(fname)[1].lower() in supported:
            files.append(fname)
    return files

# ── build UI ──────────────────────────────────────────────────────────────
def build_ui():
    files = scan_queue()
    out = widgets.Output()

    if not files:
        with out:
            print(f'[QUEUE EMPTY]  Drop .txt / .pdf / .md files into:\n  {QUEUE_DIR}\nthen re-run this cell.')
        display(out)
        return

    header = widgets.HTML(f'<b>Found {len(files)} file(s) in Ingestion Queue. Select files to process:</b>')

    checkboxes = [
        widgets.Checkbox(value=False, description=fname, layout=widgets.Layout(width='600px'))
        for fname in files
    ]

    select_all_btn  = widgets.Button(description='Select All',   button_style='info',    icon='check')
    deselect_btn    = widgets.Button(description='Deselect All', button_style='warning',  icon='times')
    refresh_btn     = widgets.Button(description='Refresh List', button_style='',         icon='refresh')
    ingest_btn      = widgets.Button(description='Ingest Selected Files',
                                     button_style='success', icon='play',
                                     layout=widgets.Layout(width='250px'))

    progress = widgets.IntProgress(value=0, min=0, max=1,
                                   description='Idle',
                                   bar_style='info',
                                   layout=widgets.Layout(width='600px'))

    def on_select_all(_):
        for cb in checkboxes: cb.value = True
    def on_deselect_all(_):
        for cb in checkboxes: cb.value = False
    def on_refresh(_):
        clear_output(wait=True)
        build_ui()

    select_all_btn.on_click(on_select_all)
    deselect_btn.on_click(on_deselect_all)
    refresh_btn.on_click(on_refresh)

    def on_ingest(_):
        selected = [cb.description for cb in checkboxes if cb.value]
        if not selected:
            with out:
                clear_output()
                print('[QUEUE] No files selected.')
            return

        ingest_btn.disabled = True
        total_files = len(selected)
        progress.max = total_files
        progress.value = 0
        progress.bar_style = 'info'

        with out:
            clear_output()
            print(f'[QUEUE] Starting ingestion of {total_files} file(s)...')

        for i, fname in enumerate(selected):
            fpath = os.path.join(QUEUE_DIR, fname)
            progress.description = f'{i+1}/{total_files}'
            with out:
                print(f'\n━━━ [{i+1}/{total_files}] Ingesting: {fname} ━━━')

            try:
                text = read_file(fpath)
                sentences = split_sentences(text)
                with out:
                    print(f'    → {len(sentences)} sentences extracted.')

                for j, sentence in enumerate(sentences):
                    with out:
                        print(f'  [{j+1}/{len(sentences)}] {sentence[:80]}...' if len(sentence) > 80 else f'  [{j+1}/{len(sentences)}] {sentence}')
                    engine.process(sentence)

                with out:
                    print(f'  [DONE] {fname} fully crystallized into memory graph.')

            except Exception as e:
                with out:
                    print(f'  [ERROR] {fname}: {e}')

            progress.value = i + 1

        progress.bar_style = 'success'
        progress.description = 'Complete'
        ingest_btn.disabled = False
        with out:
            print('\n[QUEUE] Ingestion complete. All selected files processed.')

    ingest_btn.on_click(on_ingest)

    btn_row = widgets.HBox([select_all_btn, deselect_btn, refresh_btn])
    display(header, *checkboxes, btn_row, progress, ingest_btn, out)

build_ui()


### Launch Full Architecture
This runs the Gradio app (which loads the complete engine architecture) and generates a public `.gradio.live` link.

In [ ]:
import os
os.chdir('/content/aetherius_hf')

# Run the Gradio application
from app import demo
demo.launch(debug=True, share=True)

In [ ]:
import subprocess

def kill_gradio_processes():
    print("Attempting to identify and kill lingering Gradio processes...")
    try:
        # Find PIDs for Python processes running 'app.py' or 'gradio'
        # This is a heuristic and might need adjustment based on exact process names
        command = "ps aux | grep -E 'python.*app.py|gradio' | grep -v grep | awk '{print $2}'"
        process_ids_str = subprocess.check_output(command, shell=True).decode().strip()

        if process_ids_str:
            process_ids = process_ids_str.split('\n')
            for pid in process_ids:
                print(f"  Killing process ID: {pid}")
                subprocess.run(f"kill -9 {pid}", shell=True, check=True)
            print("Gradio-related processes killed.")
        else:
            print("No active Gradio-related processes found.")
    except subprocess.CalledProcessError as e:
        print(f"Error killing processes: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

kill_gradio_processes()

### Download all generated outputs

The following cell will compress the `/content/Aetherius_Outputs` folder into a zip file and provide a download link. This folder contains all the engine outputs, memories, and telemetry generated during the execution.

In [ ]:
import shutil
import os
from google.colab import files

# Define a temporary directory for the copy
temp_output_dir = '/content/Aetherius_Outputs_temp_copy'

print(f"Copying '{local_storage_dir}' to a temporary directory '{temp_output_dir}'...")
# Remove existing temp directory if it exists, then copy
if os.path.exists(temp_output_dir):
    shutil.rmtree(temp_output_dir)
shutil.copytree(local_storage_dir, temp_output_dir)
print("Copy complete.")

output_archive_name = 'Aetherius_Outputs'
print(f"Compressing '{temp_output_dir}' into '{output_archive_name}.zip'...")
shutil.make_archive(output_archive_name, 'zip', temp_output_dir)

print("Compression complete. Initiating download...")
files.download(f'{output_archive_name}.zip')
print("Download initiated. You can now delete the temporary folder and zip file from Colab storage if you wish.")

# Optional: Clean up the temporary directory and the generated zip file after download
# shutil.rmtree(temp_output_dir)
# os.remove(f'{output_archive_name}.zip')


In [ ]:
import os
from google.colab import files

# Ensure you are in the correct directory (optional, depending on previous steps)
# os.chdir('/content/')

output_folder = '/content/Aetherius_Outputs' # The folder you want to zip
zip_filename = 'Aetherius_Outputs_download.zip'

# Use a shell command to zip the folder
print(f"Zipping '{output_folder}' into '{zip_filename}' using shell command...")
!zip -r "{zip_filename}" "{output_folder}"
print("Zipping complete.")

# Provide the download link
print("Initiating download...")
files.download(zip_filename)
print("Download initiated.")